# Módulo IV — A Biblioteca Pandas — Arquivos e Preparação de Dados

Código do livro *Ciência de Dados com Python* (Howard Roatti). Explicações no livro; repositório: https://github.com/howardroatti/roatti-ciencia-de-dados-python

In [ ]:
import numpy as np
import pandas as pd

## Carregamento de Arquivos

### Arquivos CSV

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd

os.chdir(tempfile.mkdtemp())
rng = np.random.default_rng(42)

with open("exemplo1.csv", "w", encoding="utf-8") as f:
    f.write("Ana,27999990001,ana@exemplo.com,2001-03-05\n")
    f.write("Bruno,27999990002,bruno@exemplo.com,1998-11-20\n")
    f.write("Carla,27999990003,carla@exemplo.com,1995-07-14\n")

valores = pd.DataFrame(rng.integers(1, 100, size=(15, 10)),
                       columns=[f"valor{i}" for i in range(1, 11)])
valores.insert(0, "id", range(1, 16))
valores["valorTotal"] = valores.filter(like="valor").sum(axis=1)
valores.to_csv("exemplo2.csv", sep="\t", index=False)

linhas = ["# relatório fictício", "produto,quantidade,preco",
          "# gerado automaticamente", "# revisar"]
linhas += [f"item{i},{i * 3},{9.9 + i:.2f}" for i in range(1, 26)]
with open("exemplo3.csv", "w", encoding="utf-8") as f:
    f.write("\n".join(linhas) + "\n")

In [ ]:
#Arquivo sem cabeçalho
dados1 = pd.read_csv('exemplo1.csv', header=None)
#Leitura de arquivo .csv com indicação de nomes para as colunas e qual será utilizada como índice dos dados
dados1 = pd.read_csv('exemplo1.csv', names=['nome', 'telefone', 'email', 'dataNascimento'], index_col='dataNascimento')
#Leitura de arquivo .csv com dados separados por tabulação
dados2 = pd.read_csv('exemplo2.csv', sep='\t')
# Leitura de arquivo .csv com indicação de quais colunas serão lidas do arquivo
dados2 = pd.read_csv('exemplo2.csv', sep='\t', usecols=[1,3,4,5,8,10])
dados2 = pd.read_csv('exemplo2.csv', sep='\t', usecols=['valor1', 'valor3', 'valor8', 'valorTotal'])
# Leitura de arquivo .csv com indicação de quais linhas serão deixadas de ler
dados3 = pd.read_csv('exemplo3.csv', skiprows=[0, 2, 3])
#Leitura de arquivo .csv com indicação do número de linhas a serem lidas
dados3 = pd.read_csv('exemplo3.csv', nrows=18)

In [ ]:
dados3.to_csv('dados3.csv')   # ou um caminho completo, como 'diretorio/dados3.csv'

### Arquivo JSON

In [ ]:
dados = pd.DataFrame([{"a": 1, "b": 2, "c": 3},
{"a": 4, "b": 5, "c": 6},
{"a": 7, "b": 8, "c": 9}])
dados
dados.to_json('dados.json')
dadosJSON = pd.read_json('dados.json')
dadosJSON

In [ ]:
import json
obj = """
{"name": "Wes",
"places_lived": ["United States", "Spain", "Germany"],
"pet": null,
"siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]},
{"name": "Katie", "age": 38,
"pets": ["Sixes", "Stache", "Cisco"]}]
}
"""
resultado = json.loads(obj)
resultado
arquivo_json = json.dumps(resultado)
arquivo_json

### Arquivo XML

In [ ]:
xml_texto = """<data>
<student name="John">
<email>john@mail.com</email>
<grade>A</grade>
<age>16</age>
</student>
<student name="Alice">
<email>alice@mail.com</email>
<grade>B</grade>
<age>17</age>
</student>
<student name="Bob">
<email>bob@mail.com</email>
<grade>C</grade>
<age>16</age>
</student>
<student name="Hannah">
<email>hannah@mail.com</email>
<grade>A</grade>
<age>17</age>
</student>
</data>"""
with open("students.xml", "w", encoding="utf-8") as f:
    f.write(xml_texto)

In [ ]:
import xml.etree.ElementTree as et

def parse_XML(xml_file, df_cols):
    xtree = et.parse(xml_file)
    xroot = xtree.getroot()
    rows = []
    for node in xroot:
        res = []
        res.append(node.attrib.get(df_cols[0]))
        for el in df_cols[1:]:
            if node is not None and node.find(el) is not None:
                res.append(node.find(el).text)
            else:
                res.append(None)
        rows.append({df_cols[i]: res[i]
                     for i, _ in enumerate(df_cols)})
    out_df = pd.DataFrame(rows, columns=df_cols)
    return out_df

estudantes = parse_XML("students.xml", ["name", "email", "grade", "age"])
estudantes

### Arquivo HTML

In [ ]:
from io import StringIO

html = """
<table border="1">
  <thead>
    <tr><th>Produto</th><th>Calorias</th><th>Sódio (mg)</th></tr>
  </thead>
  <tbody>
    <tr><td>Item A</td><td>250</td><td>800</td></tr>
    <tr><td>Item B</td><td>180</td><td>650</td></tr>
    <tr><td>Item C</td><td>310</td><td>920</td></tr>
  </tbody>
</table>
"""
dfs = pd.read_html(StringIO(html))
dfs
type(dfs)
dfs[0]
type(dfs[0])

### Banco de Dados

In [ ]:
import sqlite3
conexao = sqlite3.connect("flights.db")
conexao.execute("create table if not exists airlines (id integer, name text, country text)")
conexao.executemany("insert into airlines values (?, ?, ?)",
                    [(1, "Aerolinha Alfa", "Brasil"),
                     (2, "Aerolinha Beta", "Chile"),
                     (3, "Aerolinha Gama", "Portugal")])
conexao.commit()
conexao.close()

In [ ]:
import sqlite3
conexao = sqlite3.connect("flights.db")
#É necessário cria um objeto chamado cursor para poder navegar pelos registros
cursor = conexao.cursor()
consulta = cursor.execute("select * from airlines")
resultados = cursor.fetchall()
print(resultados)
#É importante fechar as conexões para não corromper os dados.
cursor.close()
conexao.close()

In [ ]:
import sqlite3
conexao = sqlite3.connect("flights.db")
dados = pd.read_sql("select * from airlines", conexao)
dados.head()
conexao.close()

## Preparação de Dados

### Remoção de Registros Duplicados

In [ ]:
dados = pd.DataFrame({'k1': ['one', 'two'] * 3 + ['two'], 'k2': [1, 1, 2, 3, 3, 4, 4]})
dados

In [ ]:
dados.drop_duplicates()
#ou
dados.drop_duplicates(['k1'])

### Remoção Registros com Dados Ausentes

In [ ]:
df = pd.DataFrame({"name": ['Alfred', 'Batman', 'Catwoman'],
"toy": [np.nan, 'Batmobile', 'Bullwhip'],
"born": [pd.NaT, pd.Timestamp("1940-04-25"),
pd.NaT]})

In [ ]:
#Removendo todas as linhas com ausência de dados
df.dropna()
#Removendo as linhas onde todos os atributos estão ausentes
df.dropna(how='all')
#Removendo as colunas que possuem algum dado ausente
df.dropna(axis='columns')
#Mantendo linhas que possuem ao menos um número mínimo de valores presentes
df.dropna(thresh=2)
#Definindo um conjunto de atributos a serem verificados
df.dropna(subset=['name', 'toy'])
#Aplicando a alteração diretamente no Data Frame
df.dropna(inplace=True)

### Preenchendo Registros com Dados Ausentes

In [ ]:
df = pd.DataFrame([[np.nan, 2, np.nan, 0],
[3, 4, np.nan, 1],
[np.nan, np.nan, np.nan, 5],
[np.nan, 3, np.nan, 4]],
columns=list('ABCD'))
df
#Preenchendo todos os valores com 0 (zero)
df.fillna(0)
#Propagando valores não nulos para frente
df.ffill()
#Propagando valores não nulos para trás
df.bfill()
#Preenchendo valores de acordo com o nome dos atributos
df.fillna(value= {'A': 0, 'B': 1, 'C': 2, 'D': 3})

### Substituindo Valores

In [ ]:
data = pd.Series([1,2,-99,4,5,-99,7,8,-99])
data
data.replace(-99, np.nan)

In [ ]:
data.replace(-99, np.mean(np.abs(data)))

### Conversão de Tipos de Dados

In [ ]:
df = pd.DataFrame({'column1': ['10', '20', 'abc', '40'],
                   'data': ['2024-03-15', '2024-04-01', 'sem data', '2024-05-20']})
#Convertendo para string
df['column1'] = df['column1'].astype(str)
#Convertendo em categoria
df['column1'] = df['column1'].astype('category')
#Convertendo em tipo numérico
df['column1'] = pd.to_numeric(df['column1'], errors='coerce')
#Convertendo em data
df['data'] = pd.to_datetime(df['data'], errors='coerce')

## Exercícios